# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Access metadata as an object and print overview
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's enumerate available record sets with their `@id`, list their fields and columns, and show the first record example from each record set.

All entities (record sets, fields, columns) are referenced by their `@id` as per FAIR guidelines and to ensure reproducibility and unambiguous reference.

In [ ]:
# List all record sets by @id and their fields
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields (`@id`):")
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"    - {f['@id']}")
            elif isinstance(f, str):
                print(f"    - {f}")
    if 'column' in rs:
        cols = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print("  Columns (`@id`):")
        for col in cols:
            if isinstance(col, dict) and '@id' in col:
                print(f"    - {col['@id']}")
            elif isinstance(col, str):
                print(f"    - {col}")
    # Display first record (if available)
    try:
        first_record = next(dataset.records(record_set=rs['@id']))
        print(f"  Example record: { {k:v for k,v in list(first_record.items())[:5]} } ...\n")
    except Exception as e:
        print(f"  Could not fetch records for this set: {e}\n")

## 3. Data Extraction

We'll extract records from **all** record sets found above and load them into separate pandas DataFrames using their record set `@id`.

Refer to record set and field `@id`s from the overview when accessing and manipulating data.

In [ ]:
# List of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for Record Set: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Show columns for the first record set (if available)
if len(dataframes) > 0:
    sample_record_set_id = record_set_ids[0]
    print(f"\nColumns in '{sample_record_set_id}':")
    print(dataframes[sample_record_set_id].columns.tolist())
    dataframes[sample_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Below, we select a numeric field for analysis using its full `@id`, filter outliers, normalize the data, and group by a categorical field (referenced by its `@id`). Adjust IDs as needed based on your inspection above.

> 📝 **Tip:** If you are unsure about field/data types, use `DataFrame.info()` or `DataFrame.describe()` to explore.

In [ ]:
# Example: Select a record set to analyze (replace with the most relevant one)
record_set_id = record_set_ids[0] if record_set_ids else None

# Investigate the columns/fields in this record set
if record_set_id and dataframes[record_set_id].shape[0] > 0:
    df = dataframes[record_set_id]
    print("Available fields (@id):", list(df.columns))

    # Try to select a numeric field by @id (replace with an actual numeric field @id from your dataset)
    numeric_candidate_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidate_fields:
        numeric_field_id = numeric_candidate_fields[0]
        print(f"Using numeric field for filtering: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # or adjust
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (first 5 rows):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} (first 5 rows):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (choose a string/categorical field by @id)
        group_candidate_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
        if group_candidate_fields:
            group_field_id = group_candidate_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (first 5 groups):")
            print(grouped_df.head())
        else:
            print("No categorical fields found for grouping.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("Record set is empty or not available.")

## 5. Visualization

Let's plot a visualization (e.g., histogram of a numeric field, or bar chart of group means). You may edit the field `@id` references below to match your data exploration.


In [ ]:
import matplotlib.pyplot as plt

# Plot histogram for the numeric field (if present)
if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30, color='cornflowerblue', edgecolor='black')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

# If grouping was done, plot group means
if 'grouped_df' in locals() and not grouped_df.empty:
    grouped_df.head(10).plot(kind='bar', legend=False, figsize=(10,5))
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.show()

## 6. Conclusion

In this notebook, we:
- Loaded the FAIR<sup>2</sup> dataset using its Croissant schema with the `mlcroissant` API.
- Explored the available record sets and their field/column `@id`s for robust referencing.
- Loaded data into DataFrames for tabular exploration and summarized schema-level observations.
- Performed exploratory data analysis (EDA) such as filtering, normalization, and grouping using strictly the field and record set `@id`s.
- Visualized data distributions and relationships to support further analysis or downstream modeling.

**Remember:** All data entities are referenced by their `@id` for consistency and reproducibility within the FAIR data paradigm.